# **Setup**

In [94]:
import subprocess
import os
import sys
import itertools
import tempfile
import random
import pandas as pd
from pathlib import Path

sys.path.append(os.path.abspath("../utils"))
from preprocessing_tools import *
from calculate_g2p_accuracy import *
from alignment import align_trxn

In [95]:
current_dir = Path().resolve()

scripts_dir = current_dir.parent / "scripts"
sh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh")]

for file in sh_files:

    file_dir = scripts_dir / file

    try:
        # Read the file in binary mode
        with open(file_dir, "rb") as f:
            content = f.read()
        
        # Replace Windows line endings (\r\n) with Linux line endings (\n)
        content = content.replace(b"\r\n", b"\n")
        
        # Write the corrected content back
        with open(file_dir, "wb") as f:
            f.write(content)
            
        print("Success! Converted line endings to LF.")
    except FileNotFoundError:
        print(f"Could not find the file at {file_dir}")

Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.
Success! Converted line endings to LF.


In [96]:
paths_dir = current_dir / "paths"
paths_dir.mkdir(parents=True, exist_ok=True)

getsh_files = [f for f in os.listdir(scripts_dir) if f.endswith(".sh") and f.startswith("get-")]

for file in getsh_files[:-1]:
    try:
        # 3. Run the script
        result = subprocess.run(
            ["bash", file], 
            cwd=scripts_dir,                    
            check=True,
            capture_output=True,
            text=True
        )
        print(f"You should now see {file} inside the {paths_dir} folder.")
        
    except subprocess.CalledProcessError as e:
        print(f"Script failed with exit code {e.returncode}")
        print(f"Error details:\n{e.stderr}")
    except FileNotFoundError:
        print(f"Could not find the scripts directory at: {scripts_dir}")

You should now see get-clean-csv-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-csv-testref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-csv-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-log-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-clean-log-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-testref-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-csv-train-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.
You should now see get-log-test-files.sh inside the /workspaces/Taglog-G2P/notebook/paths folder.


The paths to the dataset are also stored into their respective arrays.

In [97]:
txt_files = [f for f in os.listdir(paths_dir) if f.endswith(".txt")]

train_files = []
test_files = []
testref_files = []

clean_train_files = []
clean_test_files = []
clean_testref_files = []

for file in txt_files:
    file_path = paths_dir / file
    # Cleaned
    if "clean" in file:
        if "train" in file:
            clean_train_files.append(file_path)
        elif "testref" in file:
            clean_testref_files.append(file_path)
        else:
            clean_test_files.append(file_path)
    # Uncleaned
    else:
        if "train" in file:
            train_files.append(file_path)
        elif "testref" in file:
            testref_files.append(file_path)
        else:
            test_files.append(file_path)


# **Building the Train, Validation and Test sets**

In [98]:
dicts_dir = current_dir / "dicts"

create_dict(train_files[1], train_files[0], dicts_dir / "train.dict")

../data/_small/train/TGL/0812.110816.021250.csv
../data/_small/train/TGL/2359.110818.085852.csv


In [99]:
dicts_dir = current_dir / "dicts"

create_dict(clean_train_files[1], clean_train_files[0], dicts_dir / "clean_train.dict")

../data/_clean/train/TGL/0812.110816.021250.csv
../data/_clean/train/TGL/2359.110818.085852.csv


In [100]:
wlist_dir = current_dir / "wlist"

create_wlist(clean_test_files[1], wlist_dir / "clean_test.wlist")

# **Baseline Model**

The baseline model will be from Aquino et al (2019), specifically their monolingual tagalog G2P.

In [101]:
result = subprocess.run(
    ["phonetisaurus-train", "--lexicon", dicts_dir / "train.dict", "--seq2_del"],
    capture_output=True,
    text=True
)

print(result.stderr)

INFO:phonetisaurus-train:2026-05-16 15:39:26:  Checking command configuration...
INFO:phonetisaurus-train:2026-05-16 15:39:26:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-05-16 15:39:26:  Aligning lexicon...
INFO:phonetisaurus-train:2026-05-16 15:39:27:  Training joint ngram model...
INFO:phonetisaurus-train:2026-05-16 15:39:27:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-05-16 15:39:27:  G2P training succeeded: train/model.fst



In [102]:
model_dir = current_dir / "train"

output_txt = current_dir / "output" / "baseline_output.txt"

with open(output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(model_dir / "model.fst"),
            "--word_list", str(wlist_dir / "clean_test.wlist"),
        ],
        stdout=f,
        text=True
    )



In [103]:
create_trxn(clean_test_files[1], output_txt)

os.makedirs("./errors", exist_ok=True)
calculate_accuracy(clean_test_files[0], clean_testref_files[0], "baseline_errors")

../data/_clean/test/TGL/0156.110816.035620.csv
../data/_clean/testref/TGL/0156.110816.035620.csv
total phones in reference: 9253
total edit distance:       429
phone error rate:          0.0463633416189344


# **Model on Clean Data**

The data is cleaned through Claude

In [104]:
result = subprocess.run(
    ["phonetisaurus-train", "--lexicon", dicts_dir / "mclean_train.dict", "--seq2_del", "--model", "clean_model"],
    capture_output=True,
    text=True
)

print(result.stderr)

INFO:phonetisaurus-train:2026-05-16 15:39:29:  Checking command configuration...
INFO:phonetisaurus-train:2026-05-16 15:39:29:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-05-16 15:39:29:  Aligning lexicon...
INFO:phonetisaurus-train:2026-05-16 15:39:31:  Training joint ngram model...
INFO:phonetisaurus-train:2026-05-16 15:39:31:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-05-16 15:39:31:  G2P training succeeded: train/clean_model.fst



In [ ]:
model_dir = current_dir / "train"

clean_output_txt = current_dir / "output" / "clean_output.txt"
clean_output_txt.touch(exist_ok=True)

with open(clean_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model", str(model_dir / "clean_model.fst"),
            "--word_list", str(wlist_dir / "clean_test.wlist"),
        ],
        stdout=f,
        text=True
    )

Traceback (most recent call last):
  File "/usr/local/bin/phonetisaurus-apply", line 347, in <module>
    tester.ApplyG2PModel (args.word_list)
  File "/usr/local/bin/phonetisaurus-apply", line 296, in ApplyG2PModel
    raise IOError("Word list file not found.")
OSError: Word list file not found.


In [106]:
create_trxn(clean_test_files[1], clean_output_txt)

os.makedirs("./errors", exist_ok=True)
calculate_accuracy(clean_test_files[0], clean_testref_files[0], "clean_errors")

../data/_clean/test/TGL/0156.110816.035620.csv
../data/_clean/testref/TGL/0156.110816.035620.csv
total phones in reference: 9253
total edit distance:       9253
phone error rate:          1.0


# **Hyperparameter Optimized Model**

Trained and tested on cleaned data.

In [107]:
best_cmd = [
    "phonetisaurus-train",
    "--lexicon",     str(dicts_dir / "train.dict"),
    "--ngram_order", str(int(4)),
    "--seq1_max",    str(int(1)),
    "--seq2_max",    str(int(2)),
    "--seq2_del",
    "--grow",
    "--model",      str(model_dir / "tuned_model")
]

print("Training command:", " ".join(best_cmd))

result = subprocess.run(best_cmd, capture_output=True, text=True)
print(result.stderr)

Training command: phonetisaurus-train --lexicon /workspaces/Taglog-G2P/notebook/dicts/train.dict --ngram_order 4 --seq1_max 1 --seq2_max 2 --seq2_del --grow --model /workspaces/Taglog-G2P/notebook/train/tuned_model
INFO:phonetisaurus-train:2026-05-16 15:39:34:  Checking command configuration...
INFO:phonetisaurus-train:2026-05-16 15:39:34:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-05-16 15:39:34:  Aligning lexicon...
INFO:phonetisaurus-train:2026-05-16 15:39:35:  Training joint ngram model...
INFO:phonetisaurus-train:2026-05-16 15:39:35:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-05-16 15:39:35:  G2P training succeeded: /workspaces/Taglog-G2P/notebook/train/tuned_model.fst



In [108]:
tuned_output_txt = current_dir / "output" / "tuned_output.txt"
tuned_output_txt.touch(exist_ok=True)

with open(tuned_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model",     str(model_dir / "tuned_model.fst"),
            "--word_list", str(wlist_dir / "clean_test.wlist"),
        ],
        stdout=f, text=True
    )

create_trxn(clean_test_files[1], tuned_output_txt)

print("Tuned model:")
calculate_accuracy(clean_test_files[0], clean_testref_files[0], "hyperparameters_errors")

Tuned model:
../data/_clean/test/TGL/0156.110816.035620.csv
../data/_clean/testref/TGL/0156.110816.035620.csv
total phones in reference: 9253
total edit distance:       461
phone error rate:          0.04982167945531179


# **Error Analysis**


In [109]:
Most common

SyntaxError: invalid syntax (314424916.py, line 1)

# **Conclusion**

The cleaned Filipino-only dataset outperformed the uncleaned baseline model of AAquino et al.; however, hyperparameter optimization did not lead to further performance improvements. These results suggest that the best-performing WFST G2P model for Filipino is one trained and evaluated on a Filipino-only dataset using the default Phonetisaurus hyperparameters.

# **Extra: Phonemes/Tokens**

In [ ]:
corpus = current_dir / "train" / "clean_model.corpus"

phonemes = set()

with open(corpus) as f:
    for line in f:
        for unit in line.strip().split():
            if "}" in unit:
                _, phone = unit.split("}", 1)

                for p in phone.split("|"):
                    if p != "_":
                        phonemes.add(p)

phonemes_txt = current_dir / "output" / "phonemes.txt"
phonemes_txt.touch(exist_ok=True)

with open(phonemes_txt, "w") as f:
    for phone in sorted(phonemes):
        f.write(phone + "\n")

### **Below lies the hyperparameter optimization**

# **Hyperparameter Optimization**

Grid search over 6 Phonetisaurus hyperparameters using 5-fold cross-validation on the training dictionary, following the methodology of Section 4.2.4. Covers 160 configurations total.

In [ ]:
# Load training dictionary
train_dict_path = dicts_dir / "train.dict"
with open(train_dict_path) as f:
    train_entries = [line.strip() for line in f if line.strip()]

print(f"Total training entries: {len(train_entries)}")

# Hyperparameter grid (Table 4.1)
param_grid = {
    "ngram_order": [3, 4, 5, 6, 7],
    "seq1_max":    [1, 2, 3, 4],
    "seq2_max":    [1, 2],
    "seq1_del":    [True, False],
    "seq2_del":    [True, False],
    "grow":        [True, False],
}

keys = list(param_grid.keys())
configs = list(itertools.product(*param_grid.values()))
print(f"Total configurations: {len(configs)}")

# 5-fold word-level split (fixed seed for reproducibility)
random.seed(42)
shuffled = train_entries.copy()
random.shuffle(shuffled)

N_FOLDS = 5
fold_size = len(shuffled) // N_FOLDS
folds = [shuffled[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += shuffled[N_FOLDS * fold_size:]  # remainder into last fold
print(f"Fold sizes: {[len(f) for f in folds]}")

Total training entries: 1667
Total configurations: 320
Fold sizes: [333, 333, 333, 333, 335]


In [ ]:
results_path = current_dir / "output" / "hyperparam_results.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)

rows = []

for config_idx, config_vals in enumerate(configs):
    config = dict(zip(keys, config_vals))
    fold_pers = []

    for k in range(N_FOLDS):
        held_out  = folds[k]
        train_fold = [e for i, fold in enumerate(folds) if i != k for e in fold]

        with tempfile.TemporaryDirectory() as tmpdir:
            tmpdir = Path(tmpdir)

            # Write fold training dict
            fold_dict = tmpdir / "fold.dict"
            fold_dict.write_text("\n".join(train_fold))

            # Write held-out word list (deduplicated, preserving order)
            seen = set()
            wlist_lines = []
            for line in held_out:
                parts = line.split()
                if parts and parts[0] not in seen:
                    seen.add(parts[0])
                    wlist_lines.append(parts[0])
            held_wlist = tmpdir / "held.wlist"
            held_wlist.write_text("\n".join(wlist_lines))

            # Build training command
            cmd = [
                "phonetisaurus-train",
                "--lexicon",     str(fold_dict),
                "--ngram_order", str(config["ngram_order"]),
                "--seq1_max",    str(config["seq1_max"]),
                "--seq2_max",    str(config["seq2_max"]),
            ]
            if config["seq1_del"]: cmd.append("--seq1_del")
            if config["seq2_del"]: cmd.append("--seq2_del")
            if config["grow"]:     cmd.append("--grow")
            try:
                train_result = subprocess.run(
                    cmd, cwd=tmpdir, capture_output=True, text=True, timeout=120
                )
            except subprocess.TimeoutExpired:
                print("TIMEOUT:", config)
                fold_pers.append(1.0)
                break

            if train_result.returncode != 0:
                fold_pers.append(1.0)  # penalise failed configs
                continue

            # Apply model to held-out words
            apply_result = subprocess.run(
                [
                    "phonetisaurus-apply",
                    "--model",     str(tmpdir / "train" / "tuned_model.fst"),
                    "--word_list", str(held_wlist),
                ],
                capture_output=True, text=True, cwd=tmpdir
            )

            fold_pers.append(compute_per_cv(held_out, apply_result.stdout))

    mean_per = sum(fold_pers) / len(fold_pers)
    rows.append({**config, "mean_per": mean_per})

    # Save incrementally so progress isn't lost
    pd.DataFrame(rows).to_csv(results_path, index=False)

    if (config_idx + 1) % 10 == 0:
        best_so_far = min(r["mean_per"] for r in rows)
        print(f"[{config_idx + 1}/{len(configs)}] best so far: {best_so_far:.4f}")

print("Grid search complete.")

TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 1, 'seq1_del': True, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 1, 'seq1_del': False, 'seq2_del': False, 'grow': True}
[10/320] best so far: 1.0000
TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 2, 'seq1_del': True, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 1, 'seq2_max': 2, 'seq1_del': False, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 1, 'seq1_del': True, 'seq2_del': False, 'grow': True}
[20/320] best so far: 1.0000
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 1, 'seq1_del': False, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 2, 'seq1_del': True, 'seq2_del': False, 'grow': True}
[30/320] best so far: 1.0000
TIMEOUT: {'ngram_order': 3, 'seq1_max': 2, 'seq2_max': 2, 'seq1_del': False, 'seq2_del': False, 'grow': True}
TIMEOUT: {'ngram_order': 3, 'seq1_max

In [ ]:
results_df = pd.read_csv(results_path).sort_values("mean_per").reset_index(drop=True)

print("Top 10 configurations:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]
print(f"\nBest config:")
for k, v in best.items():
    print(f"  {k}: {v}")

Top 10 configurations:
 ngram_order  seq1_max  seq2_max  seq1_del  seq2_del  grow  mean_per
           4         1         2     False      True  True  0.078817
           3         1         2     False      True  True  0.079057
           5         1         2     False      True  True  0.079346
           4         1         2     False      True False  0.079497
           5         1         2     False      True False  0.080025
           3         1         2     False      True False  0.081234
           5         2         2      True      True False  0.081799
           5         2         2      True      True  True  0.081799
           5         2         2      True     False False  0.081800
           5         2         2     False      True  True  0.081896

Best config:
  ngram_order: 4
  seq1_max: 1
  seq2_max: 2
  seq1_del: False
  seq2_del: True
  grow: True
  mean_per: 0.0788167864523166


## **Re-train on Full Dictionary with Best Config**

In [ ]:
best_cmd = [
    "phonetisaurus-train",
    "--lexicon",     str(dicts_dir / "train.dict"),
    "--ngram_order", str(int(best["ngram_order"])),
    "--seq1_max",    str(int(best["seq1_max"])),
    "--seq2_max",    str(int(best["seq2_max"])),
]
if best["seq1_del"]: best_cmd.append("--seq1_del")
if best["seq2_del"]: best_cmd.append("--seq2_del")
if best["grow"]:     best_cmd.append("--grow")

print("Training command:", " ".join(best_cmd))

result = subprocess.run(best_cmd, capture_output=True, text=True)
print(result.stderr)

Training command: phonetisaurus-train --lexicon /workspaces/Taglog-G2P/notebook/dicts/train.dict --ngram_order 4 --seq1_max 1 --seq2_max 2 --seq2_del --grow
INFO:phonetisaurus-train:2026-05-05 08:07:01:  Checking command configuration...
INFO:phonetisaurus-train:2026-05-05 08:07:01:  Checking lexicon for reserved characters: '}', '|', '_'...
INFO:phonetisaurus-train:2026-05-05 08:07:01:  Aligning lexicon...
INFO:phonetisaurus-train:2026-05-05 08:07:18:  Training joint ngram model...
INFO:phonetisaurus-train:2026-05-05 08:07:20:  Converting ARPA format joint n-gram model to WFST format...
INFO:phonetisaurus-train:2026-05-05 08:07:21:  G2P training succeeded: train/model.fst



## **Evaluate Tuned Model on Test Set**

In [ ]:
tuned_output_txt = current_dir / "output" / "tuned_output.txt"

with open(tuned_output_txt, "w") as f:
    subprocess.run(
        [
            "phonetisaurus-apply",
            "--model",     str(model_dir / "model.fst"),
            "--word_list", str(wlist_dir / "test.wlist"),
        ],
        stdout=f, text=True
    )

create_trxn(test_files[1], tuned_output_txt)

print("Tuned model:")
calculate_accuracy(test_files[0], testref_files[0])

Tuned model:
../data/_small/test/TGL/0156.110816.035620.csv
../data/_small/testref/TGL/0156.110816.035620.csv
total phones in reference: 11089
total edit distance:       654
phone error rate:          0.05897736495626296
